# OpenMM Binding Stability Simulation (Colab)

Runs a 50ns MD simulation on a HADDOCK-docked peptide-KEAP1 complex.

**Setup:** Run cells 1-3 once per session.  
**Run:** Edit cell 4 with your candidate, then run it.  
**Verify:** Run cell 5 after the simulation completes.

In [ ]:
# Cell 1: Install dependencies
!pip install -q openmm pdbfixer

In [ ]:
# Cell 2: Mount Google Drive and clone the repo
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/MD_Simulation_UA'):
    !git clone https://github.com/PMONESKIN/MD_Simulation_UA.git
%cd /content/MD_Simulation_UA

In [ ]:
# Cell 3: Keep Colab alive (prevents disconnect during long runs)
%%javascript
function ClickConnect(){
  console.log("Keeping alive");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
# Cell 4: Run simulation
# =====================================================
# EDIT THESE TWO LINES FOR EACH CANDIDATE:
PDB_FILE = "input_structures/R4_DPETGEDF_KEAP1_best.pdb"
RUN_NAME = "R4_DPETGEDF"
SIM_NS = 50  # simulation length in nanoseconds
# =====================================================

OUTPUT_DIR = f"/content/drive/MyDrive/MD_Results/{RUN_NAME}"

# Delete old checkpoint if starting fresh (comment out to resume)
import os
chk = f"{OUTPUT_DIR}/{RUN_NAME}_checkpoint.chk"
if os.path.exists(chk):
    os.remove(chk)
    print(f"Deleted old checkpoint: {chk}")

!python run_stability.py \
    --pdb {PDB_FILE} \
    --name {RUN_NAME} \
    --ns {SIM_NS} \
    --output {OUTPUT_DIR}

In [ ]:
# Cell 5: Verify results
# Run this AFTER the simulation completes to check everything is correct.

import os
import csv

# =====================================================
# MUST MATCH Cell 4
RUN_NAME = "R4_DPETGEDF"
SIM_NS = 50
# =====================================================

OUTPUT_DIR = f"/content/drive/MyDrive/MD_Results/{RUN_NAME}"

print(f"{'='*60}")
print(f"VERIFICATION: {RUN_NAME}")
print(f"{'='*60}\n")

# Check files exist
expected_files = {
    'Solvated PDB': f"{RUN_NAME}_solvated.pdb",
    'Trajectory': f"{RUN_NAME}_trajectory.dcd",
    'Energy log': f"{RUN_NAME}_energy.csv",
    'Final PDB': f"{RUN_NAME}_final.pdb",
}

print("--- Output Files ---")
all_present = True
for label, fname in expected_files.items():
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  [OK] {label}: {fname} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {label}: {fname}")
        all_present = False

# Parse energy log
energy_file = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_energy.csv")
if os.path.exists(energy_file):
    temps = []
    times = []
    pot_energies = []
    
    with open(energy_file) as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                temps.append(float(row.get('Temperature (K)', row.get('temperature', 0))))
                # Try different possible column names
                for time_col in ['Time (ps)', 'time', '#"Time (ps)"']:
                    if time_col in row:
                        times.append(float(row[time_col]))
                        break
                for pe_col in ['Potential Energy (kJ/mole)', 'potentialEnergy']:
                    if pe_col in row:
                        pot_energies.append(float(row[pe_col]))
                        break
            except (ValueError, KeyError):
                continue
    
    print(f"\n--- Simulation Statistics ---")
    print(f"  Frames recorded: {len(temps)}")
    
    if times:
        sim_time_ns = max(times) / 1000
        expected_ns = SIM_NS
        print(f"  Simulation time: {sim_time_ns:.1f} ns (expected: {expected_ns} ns)")
        if sim_time_ns < expected_ns * 0.95:
            print(f"  [WARNING] Simulation ended early! Only {sim_time_ns:.1f}/{expected_ns} ns completed.")
        else:
            print(f"  [OK] Full simulation completed.")
    
    if temps:
        import statistics
        avg_temp = statistics.mean(temps)
        std_temp = statistics.stdev(temps) if len(temps) > 1 else 0
        print(f"\n--- Temperature ---")
        print(f"  Average: {avg_temp:.1f} K (target: 310.15 K)")
        print(f"  Std dev: {std_temp:.1f} K")
        if abs(avg_temp - 310.15) > 5:
            print(f"  [WARNING] Temperature drift detected!")
        else:
            print(f"  [OK] Temperature stable.")
    
    if pot_energies:
        # Check energy stability (last 25% vs first 25%)
        n = len(pot_energies)
        first_q = pot_energies[:n//4]
        last_q = pot_energies[3*n//4:]
        avg_first = statistics.mean(first_q)
        avg_last = statistics.mean(last_q)
        print(f"\n--- Energy Stability ---")
        print(f"  First quarter avg PE: {avg_first:.0f} kJ/mol")
        print(f"  Last quarter avg PE:  {avg_last:.0f} kJ/mol")
        drift = abs(avg_last - avg_first) / abs(avg_first) * 100
        print(f"  Drift: {drift:.2f}%")
        if drift > 5:
            print(f"  [WARNING] Significant energy drift — system may not be equilibrated.")
        else:
            print(f"  [OK] Energy stable.")

# Final verdict
print(f"\n{'='*60}")
if all_present and temps and sim_time_ns >= SIM_NS * 0.95:
    print(f"VERDICT: Simulation completed successfully.")
    print(f"\nNext: Download results from Google Drive and analyze RMSD.")
    print(f"  Drive path: MD_Results/{RUN_NAME}/")
else:
    print(f"VERDICT: Issues detected — check warnings above.")
print(f"{'='*60}")